In [1]:
import anndata as ad
import torch
from torch import nn
from torch.utils.data import DataLoader
import pytorch_lightning as pl  # pyright: ignore[reportMissingImports]
from src.dataset import ScDataModule, ScDataset
from src.ae import LightningAE
import scanpy as sc  # pyright: ignore[reportMissingImports]


adata = ad.read_h5ad('data/Alles_filtered.h5ad')
adata.var_names_make_unique()
display(adata.X)
display(adata.obs)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 5236395 stored elements and shape (4614, 11835)>

,cell_type1,data_type,cell_ontology_class,n_genes,n_counts
cell_id,,,,,
ACTTAGCGTTTN,undifferentiated cells,raw,nan,1588,5416
ATACACCCGCCC,undifferentiated cells,raw,nan,1165,4032
TCGGCGATGTAT,undifferentiated cells,raw,nan,1134,3992
GTTCGCCGTCGA,undifferentiated cells,raw,nan,1706,8109
CTAATCGCTAGT,undifferentiated cells,raw,nan,810,2743
...,...,...,...,...,...
TTTTAAGACGGN,LVM longitudinal visc. muscle,raw,visceral muscle cell,1027,3737
TCTTCACGAAGA,LVM longitudinal visc. muscle,raw,visceral muscle cell,1548,5824
GTTGCAATGGAG,LVM longitudinal visc. muscle,raw,visceral muscle cell,1136,3099


In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adatamodule = ScDataModule(adata, "cell_type1", "LabelEncoder")
ae_model = LightningAE(n_genes=adata.X.shape[1])

trainer = pl.Trainer(
    max_epochs=50,
    accelerator="auto",
    devices="auto",
    log_every_n_steps=50,
)
trainer.fit(ae_model, adatamodule)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/miniconda3/envs/lightning/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
/opt/miniconda3/envs/lightning/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:110: A column-vector y was passed when a 1d 

/opt/miniconda3/envs/lightning/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.
/opt/miniconda3/envs/lightning/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.
/opt/miniconda3/envs/lightning/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (33) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 49: 100%|██████████| 33/33 [00:01<00:00, 24.50it/s, v_num=1, val_loss=0.124, train_loss=0.115]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 33/33 [00:01<00:00, 20.20it/s, v_num=1, val_loss=0.124, train_loss=0.115]
